# Duplicate-Aware Stacking for Heart Disease Prediction — full run on Google Colab

**Manmeet Kaur · SIT307 11.1HD**

This notebook runs all eight experiments of the study with one command (`run_all.py`) and then shows the main results.
Measured result tables, figures and numerical macros are regenerated from this run. Literature values and method specifications are documented in the project configuration and code. A CPU runtime is sufficient; execution time depends on the machine. The saved-results archive is not required for a full training run.

**Before running:** upload the repository folder to Google Drive (for example `MyDrive/SIT307_11.1HD`).

In [ ]:
# 1. Locate the repository from Google Drive
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    candidates = [p.parent for p in Path('/content/drive/MyDrive').rglob('run_all.py')
                  if (p.parent / 'src' / 'protocols.py').exists()]
except ImportError:            # running locally
    candidates = [Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()]
ROOT = candidates[0]
os.chdir(ROOT)
print('Repository:', ROOT)

In [ ]:
# 2. Install the pinned dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Audit the dataset before any training (hash, schema, missing values, duplicates)
!python run_experiment.py audit | head -40

In [ ]:
# 4. Run all eight experiments and rebuild the paper assets
!python run_all.py

## Results
The cells below only read the files written by step 4.

In [ ]:
import pandas as pd
from IPython.display import display, Image
pd.set_option('display.precision', 4)
# Part 1: reproduction versus the published Table 11
display(pd.read_csv('results/01_reproduction/tables/published_vs_measured.csv')
        .query("report_type == 'table_value'")
        .pivot(index='model', columns='metric', values=['published', 'measured']))

In [ ]:
# Protocols A (row split), B (grouped outer, row inner) and C (proposed): stacking, mean over seeds 0-29
s = pd.read_csv('results/02_protocols/summary.csv')
display(s[(s.weighting == 'row')].pivot(index='model', columns='arm', values='accuracy_mean'))

In [ ]:
# Matched C - B contrast, preprocessing arms and tuning (percentage points, corrected t-test)
c = pd.read_csv('results/07_statistics/contrasts.csv')
display(c[(c.metric == 'accuracy') & (c.weighting == 'row')][['contrast','mean_pp','sd_pp','wins','ties','losses','p_corr','holm_p']])

In [ ]:
# Nested validation: development CV estimate versus unseen-group holdout (seeds 0-9)
display(pd.read_csv('results/03_nested_validation/summary.csv'))

In [ ]:
for f in ['fig_multiplicity', 'fig_roc_seed42', 'fig_e1_seen_unseen', 'fig_mechanism',
          'fig_e2_combiner_control', 'fig_nested_validation']:
    print(f); display(Image(filename=f'paper/figs/{f}.png', width=700))

## Next steps
* Save this notebook with its outputs (**File → Save a copy in Drive**) as evidence of the run.
